In [1]:
from datasets import load_dataset
from transformers import BertTokenizer

### 1 加载编码器工具

In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
tokenizer

BertTokenizer(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [3]:
# 试编码句子, 观察输出
out = tokenizer(
    text=['从明天开始，做一个幸福的人。','喂马，劈柴，周游世界。'],
    truncation=True,
    padding='max_length',
    max_length=17,
    return_tensors='pt',
    return_length=True
)

In [4]:
for k, v in out.items():
    print(k, v.shape)

input_ids torch.Size([2, 17])
token_type_ids torch.Size([2, 17])
attention_mask torch.Size([2, 17])
length torch.Size([2])


In [5]:
# 把编码还原成句子
print(tokenizer.decode(out['input_ids'][0]))

[CLS] 从 明 天 开 始 ， 做 一 个 幸 福 的 人 。 [SEP] [PAD]


### 2 定义数据集

In [6]:
import torch

In [7]:
from datasets import load_from_disk

In [8]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, split):
        self.dataset = load_from_disk('./data/ChnSentiCorp')[split]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        text = self.dataset[i]['text']
        label = self.dataset[i]['label']
        return text, label
dataset = Dataset('train')

In [9]:
len(dataset)

9600

In [10]:
dataset[20]

('非常不错，服务很好，位于市中心区，交通方便，不过价格也高！', 1)

### 3 定义计算设备

In [11]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
device

'cuda'

In [12]:
# 更简洁的写法
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [13]:
device

'cuda'

### 4 数据整理函数

In [14]:
def collate_fn(data):
    sents = [i[0] for i in data]
    labels = [i[1] for i in data]

    # 编码
    data = tokenizer(
        text=sents,
        truncation=True,
        padding='max_length',
        max_length=500,
        return_tensors='pt',
        return_length=True
    )

    # input_ids: 编码之后的数字
    # attention_mask: 0的位置是不需要计算attention的, 1的位置表示要计算attention
    # token_type_ids: token的类型, 0表示第一个句子, 1表示第二个句子
    input_ids = data['input_ids']
    attention_mask = data['attention_mask']
    token_type_ids = data['token_type_ids']
    labels = torch.LongTensor(labels)

    # 把数据拷贝到计算设备上
    # 这个操作也可以在训练的时候做,
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    token_type_ids = token_type_ids.to(device)
    labels = labels.to(device)

    return input_ids, attention_mask, token_type_ids, labels
    

In [15]:
# 测试一下整理函数
# 先模拟一批数据

data = [
    ('你站在桥上看风险', 1),
    ('看风景的人在楼上看你', 0),
    ('明月装饰了你的窗', 1),
    ('你装饰了别人的梦', 0)
]

In [16]:
input_ids, attention_mask, token_type_ids, labels = collate_fn(data)
input_ids.shape, attention_mask.shape, token_type_ids.shape, labels.shape

(torch.Size([4, 500]),
 torch.Size([4, 500]),
 torch.Size([4, 500]),
 torch.Size([4]))

In [17]:
# 数据加载器
loader = torch.utils.data.DataLoader(dataset=dataset,
                           batch_size=8,
                           collate_fn=collate_fn,
                           shuffle=True,
                           drop_last=True,)
len(loader)

1200

In [18]:
# 查看数据样例
for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader):
    break;
input_ids.shape, attention_mask.shape, token_type_ids.shape, labels

(torch.Size([8, 500]),
 torch.Size([8, 500]),
 torch.Size([8, 500]),
 tensor([1, 0, 1, 0, 1, 0, 0, 1], device='cuda:0'))

In [19]:
# 加载预训练模型
from transformers import BertModel

pretrained = BertModel.from_pretrained('bert-base-chinese')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# 计算参数量
sum(i.numel() for i in pretrained.parameters())

102267648

In [21]:
# zero-shot
# 冻结参数
for param in pretrained.parameters():
    param.requires_grad_(False)

In [22]:
# 预训练模型试算
pretrained.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
out = pretrained(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

In [24]:
out

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 3.1905e-01,  7.8420e-01,  1.1768e-01,  ..., -9.3141e-01,
          -2.7493e-01, -7.8081e-02],
         [ 2.2178e-01,  2.8443e-01,  5.5350e-01,  ..., -1.1487e+00,
          -4.1457e-01,  4.4112e-01],
         [ 1.1060e-01, -3.5979e-01,  4.2823e-01,  ...,  3.7804e-01,
          -1.0537e+00,  5.9320e-01],
         ...,
         [ 1.7599e-01, -1.3122e-01, -3.0525e-01,  ..., -3.6164e-02,
          -1.8130e-01, -2.8299e-01],
         [-2.0625e-02, -1.3811e-01, -3.5763e-01,  ...,  9.1250e-02,
          -2.3888e-01, -3.9498e-01],
         [ 4.9099e-02, -1.2252e-01, -2.2267e-01,  ...,  6.0019e-02,
          -2.2293e-01, -2.6164e-01]],

        [[-2.4094e-01,  7.0244e-01,  6.3740e-02,  ...,  1.1487e+00,
          -2.9113e-01, -4.3723e-01],
         [ 3.2351e-01,  7.7089e-01,  1.4607e-01,  ..., -4.4976e-01,
          -3.2887e-01, -1.1514e-01],
         [ 1.2143e+00,  5.0031e-01, -1.0988e+00,  ...,  8.1870e-01,
          -1.

In [25]:
out.last_hidden_state.shape

torch.Size([8, 500, 768])

In [27]:
# 定义下游任务模型
class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(in_features=768, out_features=2)

    def forward(self, input_ids, attention_mask, token_type_ids):
        # 使用与预训练模型抽取数据特征
        with torch.no_grad():
            out = pretrained(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        # 对抽取的特征只取第一个字的结果做分类, bert中第一个字 <cls>
        out = self.fc(out.last_hidden_state[:, 0])
        out = out.softmax(dim=1)
        return out
model = Model()

# 拷贝到GPU上
model.to(device)

Model(
  (fc): Linear(in_features=768, out_features=2, bias=True)
)

In [30]:
# 试算
model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

tensor([[0.4962, 0.5038],
        [0.3674, 0.6326],
        [0.3879, 0.6121],
        [0.5249, 0.4751],
        [0.5707, 0.4293],
        [0.4498, 0.5502],
        [0.3846, 0.6154],
        [0.4944, 0.5056]], device='cuda:0', grad_fn=<SoftmaxBackward0>)

In [37]:
# 训练
from torch.optim import AdamW
from transformers.optimization import get_scheduler

def train():
    # 定义优化器
    optimizer = AdamW(model.parameters(), lr=5e-4)

    # 定义损失函数
    criterion = torch.nn.CrossEntropyLoss()

    # 定义学习率调节器
    scheduler = get_scheduler(name='linear',
                 num_warmup_steps=0,
                 num_training_steps=len(loader),
                 optimizer=optimizer)

    # 切换到训练模式
    model.train()

    # 按批次遍历训练集中的数据
    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader):
        # 模型计算
        out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

        # 计算损失
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # 输出各项数据
        if i % 10 == 0:
            out = out.argmax(dim=1)
            accuracy = (out == labels).sum().item() / len(labels)
            lr = optimizer.state_dict()['param_groups'][0]['lr']
            print(i, loss.item(), lr, accuracy)

In [38]:
train()

0 0.6879101991653442 0.0004995833333333333 0.5
10 0.688362181186676 0.0004954166666666667 0.625
20 0.6030735969543457 0.00049125 0.75
30 0.6056503057479858 0.00048708333333333335 0.75
40 0.6556367874145508 0.00048291666666666665 0.625
50 0.48372209072113037 0.00047875 0.875
60 0.4381336569786072 0.00047458333333333337 1.0
70 0.49798738956451416 0.00047041666666666667 1.0
80 0.4519761800765991 0.00046625000000000003 1.0
90 0.5751475691795349 0.00046208333333333333 0.75
100 0.43058547377586365 0.0004579166666666667 1.0
110 0.588986873626709 0.00045375 0.75
120 0.604245662689209 0.00044958333333333336 0.75
130 0.5474742650985718 0.0004454166666666667 0.875
140 0.4939213991165161 0.00044124999999999996 0.875
150 0.376930296421051 0.0004370833333333333 1.0
160 0.4233710765838623 0.0004329166666666667 1.0
170 0.4132983088493347 0.00042875000000000004 1.0
180 0.5052968263626099 0.00042458333333333334 0.875
190 0.6057184934616089 0.00042041666666666665 0.625
200 0.42333897948265076 0.00041625 

In [39]:
# 测试
def test():
    # 定义测试的数据加载器
    loader_test = torch.utils.data.DataLoader(dataset=Dataset('test'),
                               batch_size=32,
                                collate_fn=collate_fn,
                                shuffle=True,
                                drop_last=True)

    # 下游任务模型切换到测试模式
    model.eval()
    correct = 0
    total = 0

    # 按批次遍历测试集中的数据
    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader_test):
        # 计算5个批次, 不需要全部遍历
        if i == 5:
            break;

        print(i)

        # 计算
        with torch.no_grad():
            out = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       token_type_ids=token_type_ids)

        # 计算准确率
        out = out.argmax(dim=1)
        correct += (out == labels).sum().item()
        total += len(labels)
        
    print(correct / total)
        

In [40]:
test()

0
1
2
3
4
0.8875
